In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

In [2]:
# Load both datasets
df = pd.read_csv("C:\\Users\\VP678WV\\OneDrive - EY\\Documents\\Delivery_Delay\\data\\processed\\dropped_new_data.csv")
# Preview the result
print(df.head())


  payment_type  profit_per_order  sales_per_customer         category_name  \
0      PAYMENT        101.010895           195.02570  Indoor/Outdoor Games   
1     TRANSFER         85.423610           245.20793  Indoor/Outdoor Games   
2      PAYMENT        261.173770           456.55527      Cardio Equipment   
3        DEBIT        -52.374670           191.65901          Water Sports   
4        DEBIT         55.085342           187.45561          Water Sports   

  customer_segment department_name   latitude  longitude  market  \
0      Home Office        Fan Shop  41.478510 -87.972565  Europe   
1         Consumer        Fan Shop  18.281605 -66.370510   LATAM   
2         Consumer        Footwear  18.281320 -71.919000  Europe   
3         Consumer        Fan Shop  18.289013 -66.370520   LATAM   
4         Consumer        Fan Shop  18.227660 -66.370530    USCA   

      order_city  ...   shipping_mode  label  dest_latitude  dest_longitude  \
0          Viena  ...  Standard Class      

In [3]:
df.describe()

,profit_per_order,sales_per_customer,latitude,longitude,order_date,order_item_discount,order_item_discount_rate,order_item_product_price,order_item_profit_ratio,order_item_quantity,sales,order_item_total_amount,order_profit_per_order,shipping_date,label,dest_latitude,dest_longitude,store_to_order_distance_km,distance_normalized
count,15548.000000,15548.000000,15548.000000,15548.000000,15548.000000,15548.000000,15548.000000,15548.000000,15548.000000,15548.000000,15548.000000,15548.000000,15548.000000,15548.000000,15548.000000,15548.000000,15548.000000,15548.000000,15548.000000
mean,22.031370,180.528490,29.807319,-84.769481,42534.810156,20.456345,0.101553,138.113811,0.125510,2.132032,201.147909,180.691564,22.589869,42538.328422,0.334127,24.805170,-7.488855,7410.505083,0.391363
std,126.098462,112.800041,9.688883,20.381942,303.714360,21.462499,0.071125,132.889010,0.464721,1.451369,126.075854,114.095227,100.506217,303.728482,0.836826,25.097955,75.208325,4384.827255,0.231625
min,-4048.848900,7.780167,-16.679447,-158.025880,42005.035000,0.000000,0.000000,9.990000,-2.750000,1.000000,9.990000,7.492500,-1523.708421,42007.504000,-1.000000,-53.162569,-123.376494,1.716173,0.000000
25%,7.900634,104.378949,18.267029,-97.675923,42271.760250,5.199600,0.040000,50.000000,0.080000,1.000000,119.980000,103.992000,7.565418,42274.943250,0.000000,13.703658,-76.954083,3438.269795,0.181533
50%,32.532083,165.990000,33.305889,-78.743336,42537.640000,14.000000,0.097286,59.990000,0.280000,1.000000,199.920000,163.991800,31.957603,42540.629000,1.000000,32.182598,1.596393,7126.782500,0.376376
75%,64.798628,245.960000,39.191764,-66.370580,42790.788000,29.992500,0.160000,199.990000,0.360000,3.000000,299.950000,245.975400,64.878294,42794.428500,1.000000,44.017584,29.932561,10137.887955,0.535435
max,629.981260,1660.293500,48.781933,114.937840,43131.793000,404.287475,0.250000,1617.149900,0.500000,5.000000,1617.149900,1485.000000,626.400000,43135.860000,1.000000,64.563385,178.020649,18932.454348,1.000000


In [5]:
cols_to_drop = [
    "latitude", "longitude",                         # raw location data → already captured via distance_normalized
    "dest_latitude", "dest_longitude",               # destination coords → redundant after distance calc
    "store_to_order_distance_km",                    # normalized version used → raw not needed
    "sales",                                         # sales = price × quantity → redundant
    "sales_per_customer",                            # customer-level info → not directly delay-related
    "profit_per_order",                              # order-level profit → already keeping better version
    "order_item_discount",                           # raw discount → discount rate more informative
    "order_item_profit_ratio",                       # overlaps with profit column → redundant
    "order_item_total_amount",                       # can be derived from price × quantity - discount
    "customer_country",                              # is_international derived so not needed anymore
    "order_status"
    # Removed 'order_status' feature because:
# - It is a high-level business process label (e.g., COMPLETE, CANCELED, PENDING)
# - It may **leak information about delivery outcome** (e.g., CANCELED may imply no delivery)
# - It is **not a causal or predictive feature** for forecasting delays in real time
# - It can introduce **data leakage** if populated after shipment events
# - Retaining it may lead to **overfitting** and overly optimistic performance
]

# Drop those columns
df = df.drop(columns=cols_to_drop)

# Check remaining columns
print("Remaining columns:", df.columns.tolist())

Remaining columns: ['payment_type', 'category_name', 'customer_segment', 'department_name', 'market', 'order_city', 'order_country', 'order_date', 'order_item_discount_rate', 'order_item_product_price', 'order_item_quantity', 'order_profit_per_order', 'order_region', 'order_state', 'product_name', 'shipping_date', 'shipping_mode', 'label', 'distance_normalized', 'is_international']


In [6]:
df.shape

(15548, 20)

In [7]:
# df["order_status"].value_counts(normalize=True) * 100
# df["order_status"].value_counts()

In [8]:
# df[df['order_status'] == 'CLOSED'][['label']].value_counts()

In [9]:
# Remove orders with 'CLOSED' status
# These are likely cancelled or failed orders and do not have delivery info,
# so they are not useful for delivery prediction or delay modeling.

# df = df[df['order_status'].str.upper() != 'CLOSED']

In [10]:
# Step 1: Convert Excel float to datetime
df["order_date"] = pd.to_datetime("1899-12-30") + pd.to_timedelta(df["order_date"], unit="D")
df["shipping_date"]  = pd.to_datetime("1899-12-30") + pd.to_timedelta(df["shipping_date"], unit="D")

df['order_weekofyear'] = df['order_date'].dt.isocalendar().week
df['order_month'] = df['order_date'].dt.month
df['order_hour'] = df['order_date'].dt.hour
df['shipping_day_of_week'] = df['shipping_date'].dt.dayofweek
# Monday = 0, Sunday = 6

# | order\_date         | shipping\_date      | is\_international | order\_weekofyear | order\_month | order\_hour | shipping\_day\_of\_week |
# | ------------------- | ------------------- | ----------------- | ----------------- | ------------ | ----------- | ----------------------- |
# | 2015-12-01 17:54:14 | 2015-12-03 08:10:00 | True              | 49                | 12           | 17          | 3                       |


# Comment: 
# df["order_dt"].head()
# Output: datetime values like 2015-11-15 17:54:14

# Step 2: Difference in minutes
df["order_shipping_time"] = (df["shipping_date"] - df["order_date"]).dt.total_seconds() / 60

# Step 3 (optional): Also in hours and days
df["order_shipping_time"] = df["order_shipping_time"] / 60
# df["delay_day"] = df["delay_hr"] / 24

# Comment:
# delay_min → 480.0 → 8 hours
# delay_day → 0.33 → 1/3rd of a day


In [11]:
from sklearn.preprocessing import MinMaxScaler

# Step 3: Drop original datetime columns
df.drop(["order_date", "shipping_date"], axis=1, inplace=True)

In [12]:
print(df.shape)
print(df.columns.tolist())

(15548, 23)
['payment_type', 'category_name', 'customer_segment', 'department_name', 'market', 'order_city', 'order_country', 'order_item_discount_rate', 'order_item_product_price', 'order_item_quantity', 'order_profit_per_order', 'order_region', 'order_state', 'product_name', 'shipping_mode', 'label', 'distance_normalized', 'is_international', 'order_weekofyear', 'order_month', 'order_hour', 'shipping_day_of_week', 'order_shipping_time']


In [13]:
# Get unique values count and list for each column
for col in df.columns:
    unique_vals = df[col].unique()
    print(f"Column: {col}")
    print(f"Unique Count: {len(unique_vals)}")
    print(f"Sample Values: {unique_vals[:5]}")  # show first 5 unique values
    print("-" * 40)

Column: payment_type
Unique Count: 4
Sample Values: ['PAYMENT' 'TRANSFER' 'DEBIT' 'CASH']
----------------------------------------
Column: category_name
Unique Count: 50
Sample Values: ['Indoor/Outdoor Games' 'Cardio Equipment' 'Water Sports'
 'Camping & Hiking' 'Cleats']
----------------------------------------
Column: customer_segment
Unique Count: 3
Sample Values: ['Home Office' 'Consumer' 'Corporate']
----------------------------------------
Column: department_name
Unique Count: 11
Sample Values: ['Fan Shop' 'Footwear' 'Apparel' 'Discs Shop' 'Outdoors']
----------------------------------------
Column: market
Unique Count: 5
Sample Values: ['Europe' 'LATAM' 'USCA' 'Pacific Asia' 'Africa']
----------------------------------------
Column: order_city
Unique Count: 2755
Sample Values: ['Viena' 'Buenos Aires' 'Bruges' 'Rancagua' 'New York City']
----------------------------------------
Column: order_country
Unique Count: 149
Sample Values: ['Austria' 'Argentina' 'Belgium' 'Chile' 'United

In [14]:
# One-hot encode categorical columns using pd.get_dummies()
# 
# Setting drop_first=True drops the first category from each column to avoid
# the "dummy variable trap", which can cause multicollinearity in linear models.
# 
# For example, if 'payment_type' has values: ['PAYMENT', 'TRANSFER', 'DEBIT', 'CASH'],
# using drop_first=True will drop 'PAYMENT' and only create binary columns for:
# 'TRANSFER', 'DEBIT', and 'CASH'.
# 
# This allows the model to infer the dropped category (PAYMENT) when all other columns are 0.
# It’s especially important for models like linear or logistic regression.
# 
# Use drop_first=False if you're using tree-based models (like Random Forest or XGBoost),
# as they are not affected by multicollinearity and may benefit from full encoding.

categorical_cols = ['payment_type', 'customer_segment', 'market', 'shipping_mode']
df = pd.get_dummies(df, columns=categorical_cols, drop_first=True)

In [15]:
print(df.shape)
df.columns.tolist()

(15548, 31)


['category_name',
 'department_name',
 'order_city',
 'order_country',
 'order_item_discount_rate',
 'order_item_product_price',
 'order_item_quantity',
 'order_profit_per_order',
 'order_region',
 'order_state',
 'product_name',
 'label',
 'distance_normalized',
 'is_international',
 'order_weekofyear',
 'order_month',
 'order_hour',
 'shipping_day_of_week',
 'order_shipping_time',
 'payment_type_DEBIT',
 'payment_type_PAYMENT',
 'payment_type_TRANSFER',
 'customer_segment_Corporate',
 'customer_segment_Home Office',
 'market_Europe',
 'market_LATAM',
 'market_Pacific Asia',
 'market_USCA',
 'shipping_mode_Same Day',
 'shipping_mode_Second Class',
 'shipping_mode_Standard Class']

In [16]:
from sklearn.preprocessing import MinMaxScaler

# Columns to normalize
num_cols = [
    'order_item_discount_rate', 'order_item_product_price',
    'order_item_quantity', 'order_profit_per_order',
    'distance_normalized', 'order_shipping_time',
    'is_international', 'order_weekofyear', 'order_month',
    'order_hour', 'shipping_day_of_week'
]

# Fit and transform
scaler = MinMaxScaler()
df[num_cols] = scaler.fit_transform(df[num_cols])

In [17]:
def frequency_encode_columns(df, columns):
    df_encoded = df.copy()
    for col in columns:
        freq = df_encoded[col].value_counts(normalize=True)
        df_encoded[col] = df_encoded[col].map(freq)
    return df_encoded

# Example usage:
freq_cols = ['order_city', "order_region",
             'order_country', 'order_state']
df = frequency_encode_columns(df, freq_cols)

In [18]:
df.columns.tolist()

['category_name',
 'department_name',
 'order_city',
 'order_country',
 'order_item_discount_rate',
 'order_item_product_price',
 'order_item_quantity',
 'order_profit_per_order',
 'order_region',
 'order_state',
 'product_name',
 'label',
 'distance_normalized',
 'is_international',
 'order_weekofyear',
 'order_month',
 'order_hour',
 'shipping_day_of_week',
 'order_shipping_time',
 'payment_type_DEBIT',
 'payment_type_PAYMENT',
 'payment_type_TRANSFER',
 'customer_segment_Corporate',
 'customer_segment_Home Office',
 'market_Europe',
 'market_LATAM',
 'market_Pacific Asia',
 'market_USCA',
 'shipping_mode_Same Day',
 'shipping_mode_Second Class',
 'shipping_mode_Standard Class']

In [19]:
columns_to_remove = [
    # Frequency encoded (originals)
    # 'order_city',
    # 'order_region',
    # 'order_country',
    # 'order_state',

#  'order_city_freq_enc',
#  'order_region_freq_enc',
#  'order_country_freq_enc',
#  'order_state_freq_enc'
    # One-hot encoded (originals)
    # 'payment_type',
    # 'customer_segment',
    # 'market',
    # 'order_status',
    # 'shipping_mode',

    # # Target mean encoded (if done)
    # 'product_name',
    # 'category_name',
    # 'department_name'
]

# Drop those columns
df = df.drop(columns=columns_to_remove)

# Check remaining columns
print("Remaining columns:", df.columns.tolist())

Remaining columns: ['category_name', 'department_name', 'order_city', 'order_country', 'order_item_discount_rate', 'order_item_product_price', 'order_item_quantity', 'order_profit_per_order', 'order_region', 'order_state', 'product_name', 'label', 'distance_normalized', 'is_international', 'order_weekofyear', 'order_month', 'order_hour', 'shipping_day_of_week', 'order_shipping_time', 'payment_type_DEBIT', 'payment_type_PAYMENT', 'payment_type_TRANSFER', 'customer_segment_Corporate', 'customer_segment_Home Office', 'market_Europe', 'market_LATAM', 'market_Pacific Asia', 'market_USCA', 'shipping_mode_Same Day', 'shipping_mode_Second Class', 'shipping_mode_Standard Class']


In [20]:
df.head()

,category_name,department_name,order_city,order_country,order_item_discount_rate,order_item_product_price,order_item_quantity,order_profit_per_order,order_region,order_state,...,payment_type_TRANSFER,customer_segment_Corporate,customer_segment_Home Office,market_Europe,market_LATAM,market_Pacific Asia,market_USCA,shipping_mode_Same Day,shipping_mode_Second Class,shipping_mode_Standard Class
0,Indoor/Outdoor Games,Fan Shop,0.007782,0.010226,0.12,0.024882,0.75,0.751056,0.163236,0.007782,...,False,False,True,True,False,False,False,False,False,True
1,Indoor/Outdoor Games,Fan Shop,0.005660,0.011513,0.04,0.024882,1.00,0.746637,0.088114,0.005660,...,True,False,False,False,True,False,False,False,False,True
2,Cardio Equipment,Footwear,0.000193,0.004952,0.24,0.055999,1.00,0.815766,0.163236,0.000257,...,False,False,False,True,False,False,False,False,False,True
3,Water Sports,Fan Shop,0.000515,0.004502,0.12,0.118221,0.00,0.691170,0.088114,0.000515,...,False,False,False,False,True,False,False,False,False,True
4,Water Sports,Fan Shop,0.013828,0.138603,0.24,0.118221,0.00,0.735770,0.042063,0.017366,...,False,False,False,False,False,False,True,False,False,True


In [21]:
df.shape

(15548, 31)

In [22]:
import pandas as pd
import numpy as np
from sklearn.model_selection import KFold

def target_mean_encoding_oof(df, col, target, n_splits=5, smoothing=10):
    df = df.copy()
    kf = KFold(n_splits=n_splits, shuffle=True, random_state=42)
    oof_encoded = pd.Series(index=df.index, dtype=float)
    global_mean = df[target].mean()

    for train_idx, val_idx in kf.split(df):
        train_fold = df.iloc[train_idx]
        val_fold = df.iloc[val_idx]

        agg = train_fold.groupby(col)[target].agg(['mean', 'count'])
        smooth = (agg['mean'] * agg['count'] + global_mean * smoothing) / (agg['count'] + smoothing)

        val_encoded = val_fold[col].map(smooth)
        val_encoded.fillna(global_mean, inplace=True)
        oof_encoded.iloc[val_idx] = val_encoded

    df[f"{col}_encoded"] = oof_encoded

    # Final mapping to apply to test later
    full_agg = df.groupby(col)[target].agg(['mean', 'count'])
    final_mapping = (full_agg['mean'] * full_agg['count'] + global_mean * smoothing) / (full_agg['count'] + smoothing)

    return df, final_mapping.to_dict()

def target_mean_encoding_multiple_columns(df, columns, target, n_splits=5, smoothing=10):
    df_encoded = df.copy()
    all_mappings = {}
    for col in columns:
        print(f"Encoding column: {col}")
        df_encoded, mapping = target_mean_encoding_oof(df_encoded, col, target, n_splits, smoothing)
        all_mappings[col] = mapping
    return df_encoded, all_mappings

def apply_target_encoding_to_test(test_df, mappings_dict, train_target_mean):
    test_df = test_df.copy()
    for col, mapping in mappings_dict.items():
        test_df[f"{col}_encoded"] = test_df[col].map(mapping)
        test_df[f"{col}_encoded"].fillna(train_target_mean, inplace=True)
    return test_df

In [23]:
from sklearn.model_selection import train_test_split

# 1. Split main dataset into train/test
train_df, test_df = train_test_split(df, test_size=0.2, random_state=42, stratify=df['label'])

cols_to_encode = ['product_name', 'category_name', 'department_name']
target = 'label'

# Step 1: Encode train
train_encoded, enc_mappings = target_mean_encoding_multiple_columns(train_df, cols_to_encode, target, smoothing=10)

# Step 2: Encode test
global_target_mean = train_df[target].mean()
test_encoded = apply_target_encoding_to_test(test_df, enc_mappings, global_target_mean)

Encoding column: product_name
Encoding column: category_name
Encoding column: department_name


C:\Users\VP678WV\AppData\Local\Temp\ipykernel_34136\3100151537.py:43: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  test_df[f"{col}_encoded"].fillna(train_target_mean, inplace=True)
C:\Users\VP678WV\AppData\Local\Temp\ipykernel_34136\3100151537.py:43: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always beha

In [24]:
train_encoded.head()

,category_name,department_name,order_city,order_country,order_item_discount_rate,order_item_product_price,order_item_quantity,order_profit_per_order,order_region,order_state,...,market_Europe,market_LATAM,market_Pacific Asia,market_USCA,shipping_mode_Same Day,shipping_mode_Second Class,shipping_mode_Standard Class,product_name_encoded,category_name_encoded,department_name_encoded
8181,Indoor/Outdoor Games,Fan Shop,0.014343,0.023218,0.60,0.024882,1.00,0.713605,0.046501,0.014343,...,False,True,False,False,False,False,True,0.328743,0.328743,0.336747
11948,Women's Apparel,Golf,0.000193,0.001222,0.52,0.024895,0.25,0.710773,0.163236,0.000193,...,True,False,False,False,False,False,True,0.291637,0.291637,0.312689
10663,Indoor/Outdoor Games,Fan Shop,0.006174,0.014729,0.52,0.024882,1.00,0.742035,0.129213,0.012156,...,False,True,False,False,False,True,False,0.333637,0.333637,0.336353
420,Camping & Hiking,Fan Shop,0.000257,0.004502,0.64,0.180436,0.00,0.767264,0.088114,0.000707,...,False,True,False,False,False,False,False,0.352007,0.352007,0.336747
14050,Cleats,Apparel,0.002122,0.049717,1.00,0.031111,0.25,0.721640,0.088114,0.009390,...,False,True,False,False,False,False,False,0.311987,0.312965,0.337919


In [25]:
columns_to_remove2 = [
    # # Frequency encoded (originals)
    # 'order_city',
    # 'order_region',
    # 'order_country',
    # 'order_state',

    # One-hot encoded (originals)
    # 'payment_type',
    # 'customer_segment',
    # 'market',
    # 'order_status',
    # 'shipping_mode',

    # # Target mean encoded (if done)
    'product_name',
    'category_name',
    'department_name'
]

# Drop those columns
train_encoded = train_encoded.drop(columns=columns_to_remove2)
test_encoded = test_encoded.drop(columns=columns_to_remove2)

# Check remaining columns
print("Remaining columns:", test_encoded.columns.tolist())

Remaining columns: ['order_city', 'order_country', 'order_item_discount_rate', 'order_item_product_price', 'order_item_quantity', 'order_profit_per_order', 'order_region', 'order_state', 'label', 'distance_normalized', 'is_international', 'order_weekofyear', 'order_month', 'order_hour', 'shipping_day_of_week', 'order_shipping_time', 'payment_type_DEBIT', 'payment_type_PAYMENT', 'payment_type_TRANSFER', 'customer_segment_Corporate', 'customer_segment_Home Office', 'market_Europe', 'market_LATAM', 'market_Pacific Asia', 'market_USCA', 'shipping_mode_Same Day', 'shipping_mode_Second Class', 'shipping_mode_Standard Class', 'product_name_encoded', 'category_name_encoded', 'department_name_encoded']


In [26]:
test_encoded.head()

,order_city,order_country,order_item_discount_rate,order_item_product_price,order_item_quantity,order_profit_per_order,order_region,order_state,label,distance_normalized,...,market_Europe,market_LATAM,market_Pacific Asia,market_USCA,shipping_mode_Same Day,shipping_mode_Second Class,shipping_mode_Standard Class,product_name_encoded,category_name_encoded,department_name_encoded
7031,0.000386,0.086764,0.04,0.006222,0.25,0.714925,0.163236,0.011899,-1,0.477089,...,True,False,False,False,False,False,True,0.278046,0.324708,0.366478
7490,0.006174,0.014729,0.04,0.180436,0.00,0.737672,0.129213,0.012156,-1,0.131572,...,False,True,False,False,False,False,True,0.359813,0.359813,0.344212
2169,0.000965,0.009840,0.28,0.242658,0.00,0.744895,0.048366,0.002315,-1,0.741636,...,False,False,True,False,False,False,True,0.353783,0.353783,0.344212
8740,0.000064,0.086764,0.12,0.024895,0.25,0.721298,0.163236,0.011899,-1,0.449897,...,True,False,False,False,False,False,True,0.298080,0.298080,0.320027
6324,0.000193,0.002058,0.00,0.118221,0.00,0.755173,0.004374,0.000193,0,0.581905,...,False,False,True,False,False,False,True,0.328425,0.328425,0.344212


In [27]:
train_encoded.to_csv("C:\\Users\\VP678WV\\OneDrive - EY\\Documents\\Delivery_Delay\\data\\processed\\train_encoded.csv")
test_encoded.to_csv("C:\\Users\\VP678WV\\OneDrive - EY\\Documents\\Delivery_Delay\\data\\processed\\test_encoded.csv")